# Normative Demand: Выпечка сытная, 35 пекарен

Компактный исследовательский ноутбук без разбиения на отдельные эксперименты.

Идея:
- работаем только с категорией `Выпечка сытная`
- ограничиваемся 35 пекарнями
- считаем простые и интерпретируемые опоры для норматива
- сразу смотрим таблицы и графики

In [ ]:
%matplotlib inline

from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

DAILY_PATH = ROOT / "data/processed/sku_daily_research_panel.csv"
CATEGORY_NAME = "Выпечка сытная"
N_BAKERIES = 35
CHUNKSIZE = 250_000
MIN_HISTORY_DAYS = 28

DATE_COL = "date"
BAKERY_COL = "bakery_id"
SKU_COL = "product_id"
SALES_COL = "observed_sales_qty"
DOW_COL = "dow"

USECOLS = [
    "date",
    "bakery_id",
    "bakery_name",
    "city",
    "product_id",
    "product_name",
    "category_name",
    "observed_sales_qty",
    "bakery_sales_qty_total",
    "category_sales_qty_in_bakery_day",
    "sku_sales_share_in_bakery_day",
    "sku_sales_share_in_category_day",
    "release_qty",
    "dow",
]

ROOT

## 1. Вспомогательные функции

In [ ]:
def _rolling_median(series: pd.Series, window: int = 28, min_periods: int = 7) -> pd.Series:
    return series.rolling(window=window, min_periods=min_periods).median().bfill().ffill().fillna(0.0)


def _weekday_factor(group: pd.DataFrame, value_col: str) -> pd.Series:
    weekday_mean = group.groupby(DOW_COL, observed=True)[value_col].mean()
    overall_mean = float(group[value_col].mean())
    if overall_mean <= 1e-12:
        factors = pd.Series(1.0, index=range(7), dtype=float)
    else:
        factors = (weekday_mean / overall_mean).reindex(range(7)).fillna(1.0)
        factor_mean = float(factors.mean())
        factors = factors if factor_mean <= 1e-12 else factors / factor_mean
    return factors


def _safe_corr(x: pd.Series, y: pd.Series) -> float:
    xv = pd.to_numeric(x, errors="coerce")
    yv = pd.to_numeric(y, errors="coerce")
    valid = xv.notna() & yv.notna()
    if valid.sum() < 3:
        return np.nan
    xv = xv[valid]
    yv = yv[valid]
    if xv.nunique() < 2 or yv.nunique() < 2:
        return np.nan
    return float(np.corrcoef(xv, yv)[0, 1])


def load_category_sample(
    path: str | Path,
    category_name: str,
    n_bakeries: int = 35,
    chunksize: int = 250_000,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    bakery_stats_parts = []
    for chunk in pd.read_csv(path, encoding="utf-8-sig", usecols=USECOLS, chunksize=chunksize, low_memory=True):
        chunk = chunk.loc[chunk["category_name"].astype(str) == category_name, ["bakery_id", "bakery_name", SALES_COL]].copy()
        if chunk.empty:
            continue
        chunk["bakery_id"] = chunk["bakery_id"].astype(str)
        chunk[SALES_COL] = pd.to_numeric(chunk[SALES_COL], errors="coerce").fillna(0.0)
        bakery_stats_parts.append(
            chunk.groupby(["bakery_id", "bakery_name"], as_index=False)
            .agg(rows=(SALES_COL, "size"), sales_sum=(SALES_COL, "sum"))
        )

    bakery_stats = pd.concat(bakery_stats_parts, ignore_index=True)
    bakery_stats = bakery_stats.groupby(["bakery_id", "bakery_name"], as_index=False).sum()
    bakery_stats = bakery_stats.sort_values(["sales_sum", "rows"], ascending=False).reset_index(drop=True)
    selected_bakeries = bakery_stats.head(n_bakeries)[["bakery_id", "bakery_name"]].copy()
    selected_ids = set(selected_bakeries["bakery_id"].astype(str))

    parts = []
    for chunk in pd.read_csv(path, encoding="utf-8-sig", usecols=USECOLS, chunksize=chunksize, low_memory=True):
        chunk = chunk.loc[chunk["category_name"].astype(str) == category_name].copy()
        if chunk.empty:
            continue
        chunk["bakery_id"] = chunk["bakery_id"].astype(str)
        chunk = chunk.loc[chunk["bakery_id"].isin(selected_ids)].copy()
        if chunk.empty:
            continue
        chunk["product_id"] = chunk["product_id"].astype(str)
        chunk[DATE_COL] = pd.to_datetime(chunk[DATE_COL], errors="coerce")
        chunk = chunk.dropna(subset=[DATE_COL, "bakery_id", "product_id"])
        for col in [SALES_COL, "bakery_sales_qty_total", "category_sales_qty_in_bakery_day", "sku_sales_share_in_bakery_day", "sku_sales_share_in_category_day", "release_qty"]:
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce")
        dow_numeric = pd.to_numeric(chunk[DOW_COL], errors="coerce")
        chunk[DOW_COL] = dow_numeric.fillna(chunk[DATE_COL].dt.weekday).astype(int)
        parts.append(chunk)

    df = pd.concat(parts, ignore_index=True)
    df = df.sort_values(["bakery_id", "product_id", DATE_COL]).reset_index(drop=True)
    return df, bakery_stats


def build_pair_profile(df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for (bakery_id, product_id), group in df.groupby(["bakery_id", "product_id"], observed=True):
        observed = pd.to_numeric(group[SALES_COL], errors="coerce").fillna(0.0)
        release = pd.to_numeric(group["release_qty"], errors="coerce").fillna(0.0)
        bakery_total = pd.to_numeric(group["bakery_sales_qty_total"], errors="coerce")
        category_total = pd.to_numeric(group["category_sales_qty_in_bakery_day"], errors="coerce")
        share_bakery = pd.to_numeric(group["sku_sales_share_in_bakery_day"], errors="coerce")
        share_category = pd.to_numeric(group["sku_sales_share_in_category_day"], errors="coerce")
        records.append({
            "bakery_id": bakery_id,
            "product_id": product_id,
            "bakery_name": group["bakery_name"].iloc[0],
            "product_name": group["product_name"].iloc[0],
            "city": group["city"].iloc[0],
            "n_days": int(len(group)),
            "date_min": group[DATE_COL].min(),
            "date_max": group[DATE_COL].max(),
            "observed_mean": float(observed.mean()),
            "zero_share": float((observed <= 0).mean()),
            "corr_release": _safe_corr(observed, release),
            "corr_bakery_total": _safe_corr(observed, bakery_total),
            "corr_category_total": _safe_corr(observed, category_total),
            "share_bakery_mean": float(share_bakery.mean()) if share_bakery.notna().any() else np.nan,
            "share_bakery_cv": float(share_bakery.std(ddof=0) / share_bakery.mean()) if share_bakery.notna().any() and float(share_bakery.mean()) > 1e-12 else np.nan,
            "share_category_mean": float(share_category.mean()) if share_category.notna().any() else np.nan,
            "share_category_cv": float(share_category.std(ddof=0) / share_category.mean()) if share_category.notna().any() and float(share_category.mean()) > 1e-12 else np.nan,
        })
    return pd.DataFrame.from_records(records)


def build_simple_normatives(df: pd.DataFrame) -> pd.DataFrame:
    parts = []
    bakery_daily = (
        df.groupby([DATE_COL, "bakery_id", DOW_COL], observed=True, as_index=False)
        .agg(bakery_sales_qty_total=("bakery_sales_qty_total", "max"), category_sales_qty_in_bakery_day=("category_sales_qty_in_bakery_day", "max"))
    )
    bakery_daily_map = bakery_daily[[DATE_COL, "bakery_id", "bakery_sales_qty_total", "category_sales_qty_in_bakery_day"]].copy()

    for _, group in df.groupby(["bakery_id", "product_id"], observed=True, sort=False):
        group = group.sort_values(DATE_COL).copy()
        observed = pd.to_numeric(group[SALES_COL], errors="coerce").fillna(0.0)
        release = pd.to_numeric(group["release_qty"], errors="coerce").fillna(0.0)
        weekday_factor = _weekday_factor(group, SALES_COL)
        observed_trend = _rolling_median(observed)
        release_trend = _rolling_median(release)
        share_bakery = _rolling_median(pd.to_numeric(group["sku_sales_share_in_bakery_day"], errors="coerce").fillna(0.0))
        share_category = _rolling_median(pd.to_numeric(group["sku_sales_share_in_category_day"], errors="coerce").fillna(0.0))

        group["norm_sales_weekday"] = (observed_trend * group[DOW_COL].map(weekday_factor).astype(float)).clip(lower=0.0)
        group["norm_release_weekday"] = (release_trend * group[DOW_COL].map(weekday_factor).astype(float)).clip(lower=0.0)
        group["norm_blend_50_50"] = (0.5 * group["norm_sales_weekday"] + 0.5 * group["norm_release_weekday"]).clip(lower=0.0)
        group["norm_bakery_share"] = (pd.to_numeric(group["bakery_sales_qty_total"], errors="coerce").fillna(0.0) * share_bakery).clip(lower=0.0)
        group["norm_category_share"] = (pd.to_numeric(group["category_sales_qty_in_bakery_day"], errors="coerce").fillna(0.0) * share_category).clip(lower=0.0)
        parts.append(group)

    return pd.concat(parts, ignore_index=True).sort_values(["bakery_id", "product_id", DATE_COL]).reset_index(drop=True)


def score_normatives(df: pd.DataFrame) -> pd.DataFrame:
    candidate_cols = [
        "norm_sales_weekday",
        "norm_release_weekday",
        "norm_blend_50_50",
        "norm_bakery_share",
        "norm_category_share",
    ]
    records = []
    for (bakery_id, product_id), group in df.groupby(["bakery_id", "product_id"], observed=True):
        observed = pd.to_numeric(group[SALES_COL], errors="coerce").fillna(0.0)
        base = {
            "bakery_id": bakery_id,
            "product_id": product_id,
            "bakery_name": group["bakery_name"].iloc[0],
            "product_name": group["product_name"].iloc[0],
            "n_days": int(len(group)),
            "observed_mean": float(observed.mean()),
        }
        for col in candidate_cols:
            candidate = pd.to_numeric(group[col], errors="coerce").fillna(0.0)
            base[f"{col}_mean"] = float(candidate.mean())
            base[f"{col}_ratio"] = float(candidate.mean() / observed.mean()) if float(observed.mean()) > 1e-12 else np.nan
            base[f"{col}_corr_observed"] = _safe_corr(candidate, observed)
            base[f"{col}_corr_release"] = _safe_corr(candidate, group["release_qty"])
            base[f"{col}_corr_bakery_total"] = _safe_corr(candidate, group["bakery_sales_qty_total"])
        records.append(base)
    return pd.DataFrame.from_records(records)


def plot_pair(df: pd.DataFrame, bakery_id: str, product_id: str, tail_days: int | None = 90) -> None:
    bakery_id = str(bakery_id)
    product_id = str(product_id)
    plot_df = df.loc[(df["bakery_id"].astype(str) == bakery_id) & (df["product_id"].astype(str) == product_id)].copy()
    if plot_df.empty:
        print("No rows found")
        return
    plot_df = plot_df.sort_values(DATE_COL)
    if tail_days is not None:
        plot_df = plot_df.tail(tail_days)

    fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
    axes[0].plot(plot_df[DATE_COL], plot_df[SALES_COL], label="observed", linewidth=2)
    axes[0].plot(plot_df[DATE_COL], plot_df["norm_sales_weekday"], label="norm_sales_weekday", linewidth=1.6)
    axes[0].plot(plot_df[DATE_COL], plot_df["norm_release_weekday"], label="norm_release_weekday", linewidth=1.6)
    axes[0].plot(plot_df[DATE_COL], plot_df["norm_bakery_share"], label="norm_bakery_share", linewidth=1.6)
    axes[0].plot(plot_df[DATE_COL], plot_df["norm_category_share"], label="norm_category_share", linewidth=1.6)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].set_title(f"bakery_id={bakery_id}, product_id={product_id}")

    axes[1].plot(plot_df[DATE_COL], plot_df["release_qty"], label="release_qty", linewidth=1.8)
    axes[1].plot(plot_df[DATE_COL], plot_df["bakery_sales_qty_total"], label="bakery_total", linewidth=1.5)
    axes[1].plot(plot_df[DATE_COL], plot_df["category_sales_qty_in_bakery_day"], label="category_total", linewidth=1.5)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


## 2. Загружаем выборку

In [ ]:
sample_df, bakery_stats = load_category_sample(
    DAILY_PATH,
    category_name=CATEGORY_NAME,
    n_bakeries=N_BAKERIES,
    chunksize=CHUNKSIZE,
)

print(f"rows: {len(sample_df):,}")
print(f"bakeries: {sample_df['bakery_id'].nunique():,}")
print(f"products: {sample_df['product_id'].nunique():,}")
display(bakery_stats.head(35))

## 3. Профиль пар bakery x SKU

In [ ]:
pair_profile = build_pair_profile(sample_df)
display(pair_profile.head())
display(pair_profile[["n_days", "observed_mean", "zero_share", "corr_release", "corr_bakery_total", "corr_category_total"]].describe())

## 4. Фильтр на нормальную длину истории

In [ ]:
eligible_pairs = pair_profile.loc[pair_profile["n_days"] >= MIN_HISTORY_DAYS, ["bakery_id", "product_id"]].copy()
eligible_df = sample_df.merge(eligible_pairs, on=["bakery_id", "product_id"], how="inner")
eligible_profile = pair_profile.merge(eligible_pairs, on=["bakery_id", "product_id"], how="inner")

print(f"eligible rows: {len(eligible_df):,}")
print(f"eligible pairs: {len(eligible_profile):,}")
display(eligible_profile[["n_days", "observed_mean", "zero_share"]].describe())

## 5. Считаем несколько простых нормативов

In [ ]:
norm_df = build_simple_normatives(eligible_df)
norm_scores = score_normatives(norm_df)

summary = pd.DataFrame({
    "candidate": [
        "norm_sales_weekday",
        "norm_release_weekday",
        "norm_blend_50_50",
        "norm_bakery_share",
        "norm_category_share",
    ],
    "mean_ratio": [
        norm_scores["norm_sales_weekday_ratio"].mean(),
        norm_scores["norm_release_weekday_ratio"].mean(),
        norm_scores["norm_blend_50_50_ratio"].mean(),
        norm_scores["norm_bakery_share_ratio"].mean(),
        norm_scores["norm_category_share_ratio"].mean(),
    ],
    "mean_corr_observed": [
        norm_scores["norm_sales_weekday_corr_observed"].mean(),
        norm_scores["norm_release_weekday_corr_observed"].mean(),
        norm_scores["norm_blend_50_50_corr_observed"].mean(),
        norm_scores["norm_bakery_share_corr_observed"].mean(),
        norm_scores["norm_category_share_corr_observed"].mean(),
    ],
    "mean_corr_release": [
        norm_scores["norm_sales_weekday_corr_release"].mean(),
        norm_scores["norm_release_weekday_corr_release"].mean(),
        norm_scores["norm_blend_50_50_corr_release"].mean(),
        norm_scores["norm_bakery_share_corr_release"].mean(),
        norm_scores["norm_category_share_corr_release"].mean(),
    ],
    "mean_corr_bakery_total": [
        norm_scores["norm_sales_weekday_corr_bakery_total"].mean(),
        norm_scores["norm_release_weekday_corr_bakery_total"].mean(),
        norm_scores["norm_blend_50_50_corr_bakery_total"].mean(),
        norm_scores["norm_bakery_share_corr_bakery_total"].mean(),
        norm_scores["norm_category_share_corr_bakery_total"].mean(),
    ],
})

display(summary.sort_values("mean_corr_observed", ascending=False))

## 6. Какие пары смотреть глазами

In [ ]:
view_df = norm_scores.merge(
    eligible_profile[["bakery_id", "product_id", "corr_release", "corr_bakery_total", "corr_category_total", "share_bakery_cv", "share_category_cv"]],
    on=["bakery_id", "product_id"],
    how="left",
)

print("Top release-anchored pairs")
display(view_df.sort_values(["corr_release", "n_days", "observed_mean"], ascending=[False, False, False]).head(10))

print("Top bakery-anchored pairs")
display(view_df.sort_values(["corr_bakery_total", "n_days", "observed_mean"], ascending=[False, False, False]).head(10))

print("Top category-anchored pairs")
display(view_df.sort_values(["corr_category_total", "n_days", "observed_mean"], ascending=[False, False, False]).head(10))

## 7. График по конкретной паре

In [ ]:
row = view_df.sort_values(["corr_release", "n_days"], ascending=[False, False]).iloc[0]
plot_pair(norm_df, row["bakery_id"], row["product_id"], tail_days=90)
row[["bakery_id", "product_id", "bakery_name", "product_name", "n_days", "corr_release", "corr_bakery_total", "corr_category_total"]]